# 03 — asyncio.Queue: Producer/Consumer Pattern

**Goal:** Understand asyncio.Queue, why producers and consumers must yield, and reproduce the exact buffering bug from sbot.

## How it works
- await queue.get() — waits until an item is available (suspends the coroutine, not the thread)
- await queue.put(item) — adds an item (waits if queue is full)
- queue.put_nowait(item) — adds an item WITHOUT yielding (instant, but no await = no chance for consumers to run)

**Key point:** await queue.get() suspends your coroutine — the event loop is free to run other coroutines while you wait. It does NOT block the thread.

## Exercise 3.1 — Basic producer/consumer

In [ ]:
import asyncio
import time

async def producer(queue):
    for i in range(5):
        await queue.put(f"item-{i}")
        print(f"[{time.time():.2f}] Produced: item-{i}")
        await asyncio.sleep(0.5)  # simulate work between produces

async def consumer(queue):
    while True:
        item = await queue.get()  # waits here until something arrives
        print(f"[{time.time():.2f}] Consumed: {item}")
        queue.task_done()

queue = asyncio.Queue()
consumer_task = asyncio.create_task(consumer(queue))
await producer(queue)
consumer_task.cancel()

### Question 3.1
Does the consumer process each item IMMEDIATELY after the producer puts it? Or does it batch? Why?

*Your answer:*



## Exercise 3.2 — THE BUG: producer doesn't yield

What if the producer puts multiple items WITHOUT yielding (`await`)? This is exactly the sbot buffering bug.

In [ ]:
async def fast_producer(queue):
    """Puts 5 items WITHOUT yielding between them."""
    for i in range(5):
        queue.put_nowait(f"item-{i}")  # put_nowait = no await = no yield
        print(f"[{time.time():.2f}] Produced: item-{i}")
    print(f"[{time.time():.2f}] Producer done (all 5 items in queue)")

async def slow_consumer(queue):
    while True:
        item = await queue.get()
        print(f"[{time.time():.2f}] Consumed: {item}")

queue = asyncio.Queue()
consumer_task = asyncio.create_task(slow_consumer(queue))
await fast_producer(queue)
await asyncio.sleep(0.1)  # let consumer catch up
consumer_task.cancel()

### Question 3.2
Look at the timestamps. Did the consumer get to run DURING the producer's loop? Or only AFTER?

**This is the sbot bug:** The agent emits thinking + tool_call + tool_result using `put_nowait()` in rapid succession. The Telegram sender loop (consumer) can't run until the agent yields. All messages batch up.

*Your answer:*



## Exercise 3.3 — Fix attempt 1: sleep(0)

`await asyncio.sleep(0)` yields control to the event loop. But does it guarantee the consumer runs?

In [ ]:
async def yielding_producer(queue):
    for i in range(5):
        queue.put_nowait(f"item-{i}")
        print(f"[{time.time():.2f}] Produced: item-{i}")
        await asyncio.sleep(0)  # yield to event loop
    print(f"[{time.time():.2f}] Producer done")

async def consumer_v2(queue):
    while True:
        item = await queue.get()
        print(f"[{time.time():.2f}]   Consumed: {item}")

queue = asyncio.Queue()
consumer_task = asyncio.create_task(consumer_v2(queue))
await yielding_producer(queue)
await asyncio.sleep(0.1)
consumer_task.cancel()

### Question 3.3
With `sleep(0)`, does each item get consumed before the next is produced? Or not always?

*Your answer:*



## Exercise 3.4 — The REAL problem: 3 tasks competing

In sbot, there were 3 tasks: agent (producer) → router (middle) → channel (consumer). With `sleep(0)`, the event loop picks tasks in FIFO order, but with 3 tasks, one `sleep(0)` isn't enough.

In [ ]:
async def agent(q1):
    """Produces messages → puts in q1."""
    for i in range(3):
        q1.put_nowait(f"msg-{i}")
        print(f"[{time.time():.2f}] Agent: emitted msg-{i}")
        await asyncio.sleep(0)  # one yield

async def router(q1, q2):
    """Moves messages from q1 → q2."""
    while True:
        msg = await q1.get()
        await q2.put(msg)
        print(f"[{time.time():.2f}]   Router: forwarded {msg}")

async def printer(q2):
    """Final consumer — prints messages."""
    while True:
        msg = await q2.get()
        print(f"[{time.time():.2f}]     Printer: {msg}")

q1, q2 = asyncio.Queue(), asyncio.Queue()
r = asyncio.create_task(router(q1, q2))
p = asyncio.create_task(printer(q2))
await agent(q1)
await asyncio.sleep(0.2)
r.cancel(); p.cancel()

### Question 3.4
Did each message get printed BEFORE the next was emitted? Or did they batch?

With 3 tasks (agent → router → printer), one `sleep(0)` only gives the event loop one chance to schedule ONE other task. The message needs TWO hops (q1→router→q2→printer), so it takes TWO yields minimum.

**This is why the sbot async queue approach failed.** And why we switched to sync callbacks for CLI.

*Your answer:*



## Exercise 3.5 — The sbot solution: sync callback + async yield

The fix: sync callback fires IMMEDIATELY (same call stack), then yield for async consumers.

In [ ]:
# This is how sbot works now

def sync_print_callback(msg):
    """CLI: prints immediately in the same call stack."""
    print(f"[{time.time():.2f}]   CLI printed: {msg}", flush=True)

telegram_queue = asyncio.Queue()

def telegram_callback(msg):
    """Telegram: queues for async HTTP POST."""
    telegram_queue.put_nowait(msg)

async def telegram_sender():
    while True:
        msg = await telegram_queue.get()
        print(f"[{time.time():.2f}]     Telegram sent: {msg}")

async def emit(msg):
    """Sync callback + yield for async consumers."""
    sync_print_callback(msg)   # CLI: immediate
    telegram_callback(msg)     # Telegram: queued
    await asyncio.sleep(0.05)  # yield for Telegram sender to POST

sender = asyncio.create_task(telegram_sender())

for i in range(3):
    await emit(f"msg-{i}")

await asyncio.sleep(0.2)
sender.cancel()

### Question 3.5
Now both CLI and Telegram get each message one by one. Why does `sleep(0.05)` work better than `sleep(0)` for Telegram?

*Your answer:*



## Summary

| Pattern | Problem | Solution |
|---------|---------|----------|
| Producer doesn't yield | Consumer never runs, messages batch | Add `await` between emits |
| `sleep(0)` with 3+ tasks | Not enough yields for multi-hop delivery | Use sync callbacks or `sleep(>0)` |
| Sync blocking in async code | Entire event loop freezes | `run_in_executor` |

**Next notebook:** `asyncio.create_task` — running background tasks (agent loop, sender loops)